# 08_very_high_customer_attainment

Generated from `alteryx-workflows\08_very_high_customer_attainment.yxmd` by compiler 2.2.0.

## How to use this notebook

1. Review and complete the source configuration.
2. Run cells from top to bottom.
3. Inspect the output previews and validate semantic parity.

> **Important:** Generated transformations are unvalidated. Confirm schemas, row counts, business rules, and destinations before promotion.


## Runtime imports

Imports and runtime helpers required by the generated PySpark transformations.


In [ ]:
from pyspark.sql import Window, functions as F


## Source ingestion

Loads and validates source datasets before transformation.


### Source ingestion 1 of 1

Produces `orders`, `customers`, `targets`.


In [ ]:
orders = spark.createDataFrame([["C001", "West", "Online", "2026-01-12", "SKU-A-100", "1200", "4"], ["C002", "East", "Store", "2026-01-18", "SKU-B-220", "840", "2"], ["C001", "West", "Online", "2026-02-03", "SKU-A-140", "640", "1"], ["C003", "North", "Partner", "2026-02-11", "SKU-C-010", "2100", "6"], ["C004", "South", "Store", "2026-03-01", "SKU-B-220", "430", "3"], ["C002", "East", "Online", "2026-03-09", "SKU-A-100", "990", "2"], ["C005", "West", "Partner", "2026-03-21", "SKU-C-010", "150", "1"], ["C003", "North", "Store", "2026-04-02", "SKU-A-140", "1760", "5"]], ["CustomerID", "Region", "Channel", "OrderDate", "SKU", "Amount", "Units"])
customers = spark.createDataFrame([["C001", "Acme Retail", "Gold"], ["C002", "Northwind", "Silver"], ["C003", "Contoso", "Gold"], ["C004", "Fabrikam", "Bronze"], ["C005", "Adventure", "Silver"]], ["CustomerID", "CustomerName", "Tier"])
targets = spark.createDataFrame([["West", "5000"], ["East", "3000"], ["North", "4000"], ["South", "2000"]], ["Region", "TargetAmount"])


## Transformations

Applies the translated Alteryx operations in dependency order.


### Transformations 1 of 2

Produces `parse_sku_1`, `parse_sku`, `parse_dates`, `amount`, and 4 more.


In [ ]:
parse_sku_1 = orders.withColumn("SKUFamily", F.regexp_extract(F.col("SKU"), "(?i)([A-Z]+)-([0-9]+)", 1))
parse_sku = parse_sku_1.withColumn("SKUNumber", F.regexp_extract(F.col("SKU"), "(?i)([A-Z]+)-([0-9]+)", 2))
parse_dates = parse_sku.withColumn("OrderDateParsed", F.to_date(F.col("OrderDate"), "yyyy-MM-dd"))
amount = parse_dates.withColumn("AmountNum", F.col("Amount").cast('double'))
units = amount.withColumn("UnitsNum", F.col("Units").cast('double'))
unit_price = units.withColumn("UnitPrice", F.when(F.col("UnitsNum") == 0, F.lit(0)).otherwise(F.col("AmountNum") / F.col("UnitsNum")))
drop_zero_amount = unit_price.filter(F.col("AmountNum") > "0")
unique_customers = customers.dropDuplicates(["CustomerID"])


### Transformations 2 of 2

Produces `enrich_customer`, `region_actual`, `tier_actual`, `target_number`, and 4 more.


In [ ]:
enrich_customer = drop_zero_amount.join(unique_customers, (drop_zero_amount["CustomerID"] == unique_customers["CustomerID"]), 'inner').drop(unique_customers["CustomerID"])
region_actual = enrich_customer.groupBy(*["Region"]).agg(F.sum(F.col("AmountNum").cast('double')).alias("Sum_AmountNum"), F.count(F.col("AmountNum")).alias("RowCount"))
tier_actual = enrich_customer.groupBy(*["Tier"]).agg(F.sum(F.col("AmountNum").cast('double')).alias("Sum_AmountNum"), F.count(F.col("AmountNum")).alias("RowCount"))
target_number = targets.withColumn("TargetAmount", F.col("TargetAmount").cast('double'))
vs_target = region_actual.join(target_number, (region_actual["Region"] == target_number["Region"]), 'inner').drop(target_number["Region"])
attainment = vs_target.withColumn("Attainment", F.col("Sum_AmountNum") / F.col("TargetAmount"))
rank_attainment = attainment.orderBy(F.col("Attainment").desc())
long_metrics = rank_attainment.unpivot(["CustomerID", "Region"], ["Amount", "Units"], "Name", "Value")


## Output preview

Displays terminal DataFrames for review. Replace preview-only output with approved destination writes after validation.


### Output preview 1 of 1


In [ ]:
display(rank_attainment)
display(tier_actual)
display(long_metrics)
display(enrich_customer)
